In [ ]:
# Setup
import asyncio
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("✓ Environment ready")

## 1. Document Ingestion

First, let's ingest some documents into the system.

In [ ]:
from src.ingestion.pipeline import IngestionPipeline

async def ingest_documents():
    pipeline = IngestionPipeline(enable_kg=True)
    await pipeline.initialize()
    
    # Ingest from big_tech_docs directory
    docs_dir = Path("../data/big_tech_docs")
    if docs_dir.exists():
        stats = await pipeline.ingest_directory(docs_dir)
        print(f"\n✓ Ingestion Complete!")
        print(f"  Documents: {stats['documents_processed']}")
        print(f"  Chunks: {stats['chunks_created']}")
        print(f"  Duration: {stats['duration_seconds']:.2f}s")
    else:
        print(f"Directory not found: {docs_dir}")

# Run ingestion (only needed once)
# await ingest_documents()
print("Ingestion cell ready (uncomment to run)")

## 2. Query with Different RAG Variants

Let's test a query using all three RAG approaches.

In [ ]:
from src.rag_variants.naive_rag import NaiveRAG
from src.rag_variants.kg_rag import KnowledgeGraphRAG
from src.rag_variants.hybrid_rag import HybridRAG

# Test query from big_tech_curated dataset
query = "What is OpenAI's latest funding round valuation?"

async def compare_rag_variants(question: str):
    # Initialize RAG systems
    naive = NaiveRAG(top_k=5)
    kg = KnowledgeGraphRAG(top_k=5)
    hybrid = HybridRAG(top_k=5)
    
    await naive.initialize()
    await kg.initialize()
    await hybrid.initialize()
    
    print(f"\nQuestion: {question}\n")
    print("=" * 80)
    
    # Naive RAG
    print("\n1. NAIVE RAG (Vector)")
    print("-" * 80)
    naive_result = await naive.generate(question)
    print(f"Answer: {naive_result['answer']}")
    print(f"Latency: {naive_result['latency_ms']:.2f}ms")
    
    # KG RAG
    print("\n2. KG RAG (Graph)")
    print("-" * 80)
    kg_result = await kg.generate(question)
    print(f"Answer: {kg_result['answer']}")
    print(f"Latency: {kg_result['latency_ms']:.2f}ms")
    
    # Hybrid RAG
    print("\n3. HYBRID RAG (Vector + Graph)")
    print("-" * 80)
    hybrid_result = await hybrid.generate(question)
    print(f"Answer: {hybrid_result['answer']}")
    print(f"Latency: {hybrid_result['latency_ms']:.2f}ms")

# Run comparison
await compare_rag_variants(query)

## 3. Run Evaluation on Custom Dataset

Run evaluation using the big_tech_curated dataset (255 questions).

In [ ]:
from src.evaluation.orchestrator import EvaluationOrchestrator

async def run_evaluation(num_questions=10):
    orchestrator = EvaluationOrchestrator()
    
    # Run on big_tech_curated dataset
    results = await orchestrator.run_evaluation(
        dataset_name="big_tech_curated",
        num_questions=num_questions,
        output_dir=Path("../benchmarks/runs/notebook_test")
    )
    
    print(f"\n✓ Evaluation Complete!")
    print(f"  Questions: {results['num_questions']}")
    print(f"  Output: {results['output_file']}")
    return results

# Run on small subset for testing (uncomment to run)
# results = await run_evaluation(num_questions=5)
print("Evaluation cell ready (uncomment to run)")

## 4. Analyze Results

Load and visualize evaluation results.

In [ ]:
import json
from src.analysis.statistics import calculate_summary_stats
from src.analysis.visualizations import plot_score_distributions

# Load latest results
results_dir = Path("../benchmarks/runs")
result_files = sorted(results_dir.glob("*/eval_*.json"))

if result_files:
    latest_file = result_files[-1]
    print(f"Loading: {latest_file.name}\n")
    
    with open(latest_file) as f:
        data = json.load(f)
    
    # Calculate summary statistics
    stats = calculate_summary_stats(data)
    
    print("Summary Statistics:")
    print("=" * 80)
    for method in ['naive_rag', 'kg_rag', 'hybrid_rag']:
        if method in stats:
            print(f"\n{method.upper()}:")
            for dim, values in stats[method].items():
                print(f"  {dim}: {values['mean']:.2f} ± {values['std']:.2f}")
    
    # Plot distributions
    plot_score_distributions(data)
else:
    print("No results found. Run evaluation first.")

## Next Steps

- Run full evaluation: `python cli.py evaluate --dataset big_tech_curated`
- Analyze results: See `analysis_tutorial.ipynb`
- View detailed docs: See `docs/` folder

## Dataset Info

**big_tech_curated** (255 questions):
- 90 factual questions (single-hop)
- 140 multi-hop questions (2-4 hops)
- 25 analytical questions
- Source: 21 documents on AI industry